## Reading from Files

For ingesting dat, PySpark provides two main structures which are

1. The **Resilient Distribubed Dataset (RDD)**
2. DataFrame

Reading data into a data frame is done through the `DataFrameReader` object which can be accessed through `spark.read`. We can read data from `csv`, `json`, `txt`. `orc` and `parquet` are also data format especially well suited for big data processing.

- **JSON** stands JavaScript Object Notation and it is more suited for semistructured, hierarchical data where you nest data structures.
- **CSV** stands for comma separated values is suitable for structured, delimetered, by a comman, data. CSV files lack schema enforcement, meaning that they doesn't define or enforce schema for each column and that leads to potential inconsistencies. 
- **Parquet** is a columnar storage format optimized for ready-heavy operations (frequent querying). It enforeces schema definitions which helps maintain data consistency and support complex nested structured. Coutering the inefficiencies of CSV while providing the functionality of JSON.
- **ORC** stands for Optimized Row Columnar and is compets with Parquet on pretty much hte same purpose.
- **Text**
- **XML**


PySpark default to Parquet when reading and writing files


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, count, avg, col

In [2]:
spark = SparkSession.builder\
    .appName("files")\
    .master("local[*]")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/15 20:41:34 WARN Utils: Your hostname, brian-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
25/11/15 20:41:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/15 20:41:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/15 20:41:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
print(spark.read)
dir(spark.read)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_df',
 '_jreader',
 '_set_opts',
 '_spark',
 'csv',
 'format',
 'jdbc',
 'json',
 'load',
 'option',
 'options',
 'orc',
 'parquet',
 'schema',
 'table',
 'text',
 'xml']

In [4]:
df = spark.read.csv(
    '../data/ecommerce_DATASET.csv', 
    header=True,
    inferSchema=True,
) # returns a data frame

By default the header names for each column in the data frame is not inferred by the read method so we if we need to use the actual names of columns, instead of arbitrary names, we need to infer the schema automatically from the csv file by setting the `inferSchema` parameter to `True`. Also, to tell PySpark that the first line in the dataset is the columns names we need to set the `header` to `True`

In [5]:
df.printSchema()

root
 |-- �InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [4]:
df.show(5) # by default, it shows 20 rows an truncate long rows but I want to view the first 5 rows

+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
|�InvoiceNo|StockCode|         Description|Quantity|    InvoiceDate|UnitPrice|CustomerID|       Country|
+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
|    536365|   85123A|WHITE HANGING HEA...|       6|01/12/2010 8:26|     2.55|     17850|United Kingdom|
|    536365|    71053| WHITE METAL LANTERN|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
|    536365|   84406B|CREAM CUPID HEART...|       8|01/12/2010 8:26|     2.75|     17850|United Kingdom|
|    536365|   84029G|KNITTED UNION FLA...|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
|    536365|   84029E|RED WOOLLY HOTTIE...|       6|01/12/2010 8:26|     3.39|     17850|United Kingdom|
+----------+---------+--------------------+--------+---------------+---------+----------+--------------+
only showing top 5 rows


In [5]:
df.count()

541909

The `count()` method counts the rows in the dataframe. In our dataset we have 541, 909 rows.

In [6]:
df.createOrReplaceTempView("commerce")

In [7]:
number_of_different_countries = spark.sql("SELECT DISTINCT Country FROM commerce").count()
spark.sql("SELECT DISTINCT Country FROM commerce").show(number_of_different_countries)

[Stage 8:============================================>              (3 + 1) / 4]

+--------------------+
|             Country|
+--------------------+
|              Sweden|
|           Singapore|
|             Germany|
|              France|
|              Greece|
|             Belgium|
|             Finland|
|               Italy|
|                EIRE|
|           Lithuania|
|              Norway|
|               Spain|
|             Denmark|
|           Hong Kong|
|             Iceland|
|              Israel|
|     Channel Islands|
|              Cyprus|
|        Saudi Arabia|
|         Switzerland|
|United Arab Emirates|
|              Canada|
|      Czech Republic|
|             Lebanon|
|               Japan|
|              Poland|
|            Portugal|
|           Australia|
|             Austria|
|             Bahrain|
|      United Kingdom|
|         Netherlands|
|  European Community|
|               Malta|
|         Unspecified|
|                 USA|
|              Brazil|
|                 RSA|
+--------------------+



In [8]:
df.groupBy("Country").agg(count("CustomerID").alias("NumberOfCustomers"))\
    .orderBy("NumberOfCustomers", ascending=False)\
    .show(number_of_different_countries)

[Stage 11:===========================================>              (3 + 1) / 4]

+--------------------+-----------------+
|             Country|NumberOfCustomers|
+--------------------+-----------------+
|      United Kingdom|           361878|
|             Germany|             9495|
|              France|             8491|
|                EIRE|             7485|
|               Spain|             2533|
|         Netherlands|             2371|
|             Belgium|             2069|
|         Switzerland|             1877|
|            Portugal|             1480|
|           Australia|             1259|
|              Norway|             1086|
|               Italy|              803|
|     Channel Islands|              758|
|             Finland|              695|
|              Cyprus|              622|
|              Sweden|              462|
|             Austria|              401|
|             Denmark|              389|
|               Japan|              358|
|              Poland|              341|
|                 USA|              291|
|              I

In [9]:
df.columns

['�InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [17]:
# For each country, get the customers and see how many orders they have placed

df.groupBy('Country', 'CustomerID')\
    .agg(count("CustomerID").alias("NumberOfOrders"))\
    .orderBy("NumberOfOrders", ascending=False)\
    .show()

+--------------+----------+--------------+
|Country       |CustomerID|NumberOfOrders|
+--------------+----------+--------------+
|United Kingdom|17841     |7983          |
|EIRE          |14911     |5903          |
|United Kingdom|14096     |5128          |
|United Kingdom|12748     |4642          |
|United Kingdom|14606     |2782          |
|United Kingdom|15311     |2491          |
|Netherlands   |14646     |2085          |
|United Kingdom|13089     |1857          |
|United Kingdom|13263     |1677          |
|United Kingdom|14298     |1640          |
|United Kingdom|15039     |1508          |
|EIRE          |14156     |1420          |
|United Kingdom|18118     |1284          |
|United Kingdom|14159     |1212          |
|United Kingdom|14796     |1165          |
|United Kingdom|15005     |1160          |
|United Kingdom|16033     |1152          |
|United Kingdom|14056     |1128          |
|United Kingdom|14769     |1094          |
|United Kingdom|17511     |1076          |
+----------

In [20]:
spark.sql("SELECT * FROM commerce WHERE CustomerID = 17841 ORDER BY Quantity DESC")\
    .select('CustomerID', 'Country', "Description", 'Quantity', 'UnitPrice')\
    .count()
    # .show(n=100, truncate=False)

7983